# CSupply Chain Digital Twins

Welcome to the final interactive notebook of the `digital-twins` repository.

We discuss the future of **Digital Twin** technologies in Industry 4.0. 

While our previous notebooks focused on physical, spatiotemporal systems (wildfires, autonomous agents, and traffic), DT is equally powerful for **discrete network events**, such as Global Supply Chains.

### The Problem: The "Offline" Supply Chain
Many modern corporations use offline simulation models for inventory management. A factory knows it uses $60$ parts a day, and an overseas shipment of $1,000$ parts takes exactly $14$ days to arrive. The offline simulation predicts inventory will never drop below $100$ units. 

But the real world is chaotic. A labor strike at a shipping port, a canal blockage, or a massive storm can delay a cargo ship by a week. If the factory doesn't realize this until Day 14, they will stock out, the assembly line will halt, and millions of dollars will be lost.

### The Solution: A Digital Twin
A Digital Twin connects the simulation directly to live data streams (e.g., GPS tracking on cargo ships or real-time port authority APIs). 

In DT terms, this is **External Input Forecasting** combined with **Simulation-Based Prediction**. The moment the port reports a 5-day delay, the Digital Twin immediately assimilates this data, re-runs the simulation forward in time, and predicts the exact hour the factory will run out of parts—giving managers days or weeks to air-freight emergency inventory.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

### Interactive Learning: Predicting the Stockout

**The Scenario:** 
Your factory starts with $1,000$ units. You consume $60$ units per day. A resupply ship carrying $1,000$ units is scheduled to arrive on Day 14. 

**Instructions:**
1. **Actual Port Delay (Days)**: The real-world delay of the ship. Set this to `5`. The Black line (True Reality) drops below zero (a catastrophic stockout). Notice the Blue line (Offline Simulation) is completely blind to this and assumes everything is fine.
2. **Data Assimilation Day**: This is the day your Digital Twin pings the Port API, discovers the delay, and runs a forward simulation. 
   - Set it to Day `15`. It's too late. The factory already ran out of parts.
   - Slide it back to Day `4`. The Green line (Digital Twin Prediction) instantly updates. It warns you on Day 4 that you will stock out on Day 16, giving you 12 days to fix the problem!

In [ ]:
def interactive_supply_chain(port_delay_days=5, assimilation_day=4):
    # --- 1. System Parameters ---
    total_days = 30
    initial_inventory = 1000
    daily_demand = 60
    shipment_size = 1000
    expected_arrival_day = 14
    
    # The actual day the ship will arrive in the real world
    true_arrival_day = expected_arrival_day + port_delay_days
    
    days_array = np.arange(0, total_days + 1)
    
    # --- 2. Offline Simulation (No DDDS, Blind to Reality) ---
    offline_inv = np.zeros(total_days + 1)
    offline_inv[0] = initial_inventory
    for i in range(1, total_days + 1):
        offline_inv[i] = offline_inv[i-1] - daily_demand
        if i == expected_arrival_day:
            offline_inv[i] += shipment_size
            
    # --- 3. True Reality (What actually happens) ---
    true_inv = np.zeros(total_days + 1)
    true_inv[0] = initial_inventory
    for i in range(1, total_days + 1):
        true_inv[i] = true_inv[i-1] - daily_demand
        if i == true_arrival_day:
            true_inv[i] += shipment_size
            
    # --- 4. Digital Twin Prediction (DDDS) ---
    # The DT tracks reality perfectly up to the 'assimilation_day'. 
    # At 'assimilation_day', it updates its external input model (the new arrival date)
    # and simulates the rest of the month into the future.
    dt_pred_inv = np.full(total_days + 1, np.nan)
    
    # Current state at assimilation time
    current_inv = true_inv[assimilation_day]
    dt_pred_inv[assimilation_day] = current_inv
    
    # Forward Simulation from assimilation day to Day 30
    for i in range(assimilation_day + 1, total_days + 1):
        current_inv -= daily_demand
        if i == true_arrival_day:
            current_inv += shipment_size
        dt_pred_inv[i] = current_inv

    # --- 5. Plotting ---
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Plot the 0 threshold (Stockout line)
    ax.axhline(0, color='red', linestyle='-', linewidth=2, label="Critical Threshold (Stockout)")
    
    # Plot Offline Model
    ax.plot(days_array, offline_inv, 'b--', linewidth=2, alpha=0.6, label='Offline Model (Assumes no delays)')
    
    # Plot True Reality
    ax.plot(days_array, true_inv, 'k-', linewidth=2, alpha=0.3, label='True Reality (Hidden)')
    
    # Plot Digital Twin Past History (Solid Green up to assimilation day)
    ax.plot(days_array[:assimilation_day+1], true_inv[:assimilation_day+1], 'g-', linewidth=3, label='Digital Twin (Known History)')
    
    # Plot Digital Twin Future Prediction (Dashed Green after assimilation day)
    ax.plot(days_array[assimilation_day:], dt_pred_inv[assimilation_day:], 'g-o', linewidth=3, markersize=5, label='Digital Twin (Simulation-Based Prediction)')
    
    # Mark the assimilation moment
    ax.axvline(assimilation_day, color='purple', linestyle=':', linewidth=2, label=f'Data Assimilated (Day {assimilation_day})')
    
    # Logic to detect if the Digital Twin caught the stockout in time
    stockout_predicted = np.any(dt_pred_inv < 0)
    stockout_day = np.where(dt_pred_inv < 0)[0]
    
    if stockout_predicted:
        first_stockout_day = stockout_day[0]
        if assimilation_day < first_stockout_day:
            msg = f"ACTION REQUIRED: Stockout predicted on Day {first_stockout_day}! You have {first_stockout_day - assimilation_day} days to air-freight parts."
            color = 'darkorange'
        else:
            msg = f"CRITICAL FAILURE: Stockout occurred on Day {first_stockout_day}. We assimilated the data too late!"
            color = 'red'
    else:
        msg = "STATUS NORMAL: Inventory levels secure."
        color = 'green'
        
    ax.text(0.02, 0.05, msg, transform=ax.transAxes, fontsize=14, fontweight='bold', color=color, 
            bbox=dict(facecolor='white', edgecolor=color, boxstyle='round,pad=0.5', alpha=0.9))
    
    # Styling
    ax.set_title("Supply Chain Digital Twin: Real-Time Inventory Prediction", fontsize=16, fontweight='bold')
    ax.set_xlabel("Time (Days)", fontsize=12)
    ax.set_ylabel("Inventory Level (Units)", fontsize=12)
    ax.set_xlim(0, total_days)
    ax.set_ylim(-500, 2000)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

# Create the interactive widget
interact(interactive_supply_chain, 
         port_delay_days=IntSlider(value=5, min=0, max=14, step=1, description='Actual Port Delay (Days):', style={'description_width': 'initial'}, layout={'width':'600px'}),
         assimilation_day=IntSlider(value=4, min=0, max=25, step=1, description='Data Assimilation (Day):', style={'description_width': 'initial'}, layout={'width':'600px'}));